# Week 11 — Train empirical-distribution diffusion (11c)

This notebook trains a reverse-diffusion model whose target is the **empirical**
per-window latitude distribution — not the residual against the parametric
classical. The classical density acts as a *prior* via a KL regularizer
instead of a fixed pedestal. Two physical constraints (non-negativity and
integration to 1) are enforced by construction: training happens in a
standardized logit space, and sampling decodes via softmax.

## Why a separate notebook

Week-11 ablations on the residual target showed no variant beating the
classical baseline on normalized NLL, and the oracle gap pointed at the
target framing rather than at architecture: cond carries almost no
predictive information about the residual because the classical already
exploits the same inputs. Switching the target to the empirical distribution
lets cond re-explain bulk *and* fine structure jointly.

## Pipeline

1. Re-use the v2 parquet built by `11_train_and_evaluate.ipynb` (no rebuild).
2. Build `EmpiricalDistributionDataset` (Laplace-smoothed mass → mean-centered
   logits → per-bin standardization), plus a per-window classical Dirichlet
   prior for the KL term.
3. Define `EXPERIMENTS_EMP` — same `_spec` template as the residual notebook,
   starting with E0 = base cond + concat arch, with space for E1+ to add
   cond_opp / cond_traj / FiLM / Fourier / CFG.
4. Train each variant with `train_empirical_experiment`. Checkpoints are
   saved as `ckpt_emp_<name>.ckpt` so they coexist with the residual
   `ckpt_<name>.ckpt` files.
5. Per-variant diagnostic: decoded sample density vs the classical density at
   one val window. Confirms the model is producing valid simplex outputs that
   are neither pinned to classical (KL too strong) nor random (KL too weak).

Evaluation lives in `11d_evaluate_empirical.ipynb`.

In [ ]:
# Standard Week 10/11 setup: locate the repo, install if missing.
import os, subprocess, sys

# Use this path if working locally; switch to /content/butterflai in Colab.
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn.functional as F

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb

In [ ]:
# Bootstrap sys.path — same pattern as 11b. The week_10 conditioned_infrastructure
# is loaded first to use its find_week10_artifacts, then we repoint to week_11
# so subsequent imports of conditioned_infrastructure / empirical_infrastructure
# pick up the week_11 versions.
_week09_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))
_week10_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_10"))
_week11_dir = os.path.abspath(os.path.join(repo_path, "weeks", "week_11"))
for _p in (_week09_dir, _week11_dir, _week10_dir):
    if _p in sys.path:
        sys.path.remove(_p)
    sys.path.insert(0, _p)
sys.modules.pop("conditioned_infrastructure", None)

from conditioned_infrastructure import find_week10_artifacts
paths = find_week10_artifacts(extra_required=[
    "data/composite_sunspot_groups_peak_area.csv",
])
if _week11_dir in sys.path:
    sys.path.remove(_week11_dir)
sys.path.insert(0, _week11_dir)
sys.modules.pop("conditioned_infrastructure", None)
sys.modules.pop("empirical_infrastructure", None)

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    build_experiment_registry,
    resolve_experiment_stems,
)
from butterflAI_model import ButterflAIModel
from empirical_infrastructure import (
    EmpiricalDistributionDataset,
    ExtendedConditionalEmpiricalDiffusionLightning,
    sample_empirical_extended,
    train_empirical_experiment,
    load_trained_empirical_experiment,
    discover_emp_experiment_checkpoints,
    classical_mass_at_bins,
    EMP_CKPT_PREFIX,
)

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PARQUET_V2: {PARQUET_V2}")
print(f"CKPT_DIR  : {CKPT_DIR}")
print(f"device    : {device}")

In [ ]:
# Load the v2 parquet — already built by 11_00_build_v2.ipynb. Do NOT
# rebuild here. Test split is reserved for the PI; this notebook works on
# train+val only.
windows_v2 = pd.read_parquet(PARQUET_V2)
windows_v2 = windows_v2.loc[windows_v2["split"].isin(["train", "val"])].reset_index(drop=True)
windows_aug = windows_v2  # API parity with train_experiment
print(f"v2 parquet rows (train+val): {len(windows_v2)}")
print(f"  splits: {windows_v2['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v2['cycle'].unique())}")

## Experiment registry

`EXPERIMENTS_EMP` mirrors the residual `EXPERIMENTS` dict so adding cond groups,
switching to FiLM / Fourier, or enabling classifier-free guidance is a single
dict edit. Two new keys specific to the empirical target:

- `lambda_kl` — weight on `KL(p_model || p_classical)`. Default 0.1. Set to 0
  to remove the classical anchor entirely; raise toward 1–3 if generated
  distributions drift unphysically far from classical.
- `alpha_smooth`, `n_pseudo_obs` — Laplace smoothing strength on the
  empirical histogram target. Defaults `alpha=1`, `n_pseudo=30` give a mild
  ~1/3 convex blend against the uniform prior, enough to dampen single-bin
  spikes in low-count windows without flattening real bimodality.

We start with **E0 = base cond, concat arch** only; the commented templates
below show how to add cond_opp / cond_traj / FiLM / Fourier / CFG variants.
Keep variant names disjoint from the residual notebook's `EXPERIMENTS` so the
checkpoint files (`ckpt_emp_*.ckpt`) never collide with `ckpt_*.ckpt`.

In [ ]:
# Experiment specs. Same 18 variants as the residual pair (11a/11b) — the
# empirical target changes the loss, not the ablation grid.
#
# Run names are GENERATED from the specs by canonical_experiment_name(), not
# typed by hand, so a name can never disagree with the config it labels:
#
#     <cond-set>_<arch>[_four][_guid][_h###][_L#]
#
#   cond-set : "base" plus added groups in the fixed order hemi < opp < traj,
#              joined by "+"  ("trajv" = traj group + its validity mask)
#   arch     : "cat" (concat) or "film" — ALWAYS stated
#   four     : fourier=True          guid : cond_dropout_p > 0
#   h###/L#  : appended only when capacity differs from _BASE_TEMPLATE
#
# Checkpoints land at ckpt_emp_<name>.ckpt (EMP_CKPT_PREFIX), so these share
# the residual family's names without colliding with its files.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     5000,
    "lr":             1e-3,
    "batch_size":     64,
    # Empirical-target specific knobs.
    "lambda_kl":      0.1,
    "alpha_smooth":   1.0,
    "n_pseudo_obs":   30.0,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

_SPECS_EMP = [
    # ── Base conditioning (4-D cond) × mechanism ────────────────────────────
    # The full 2×2×2 over arch, Fourier lifting and guidance, holding the
    # information content fixed: how much comes from mechanism alone?
    _spec(),                                                   # base_cat  (baseline)
    _spec(arch="film"),                                        # base_film
    _spec(fourier=True),                                       # base_cat_four
    _spec(arch="film", fourier=True),                          # base_film_four
    _spec(cond_dropout_p=0.1),                                 # base_cat_guid
    _spec(arch="film", cond_dropout_p=0.1),                    # base_film_guid
    _spec(fourier=True, cond_dropout_p=0.1),                   # base_cat_four_guid
    _spec(arch="film", fourier=True, cond_dropout_p=0.1),      # base_film_four_guid

    # ── Level 1 — one new cond group at a time, concat arch ─────────────────
    _spec(consumed_keys=["cond_base", "cond_cyclehemi"],       # base+hemi_cat
          groups=["base", "cyclehemi"]),
    _spec(consumed_keys=["cond_base", "cond_opp"],             # base+opp_cat
          groups=["base", "opp"]),
    _spec(consumed_keys=["cond_base", "cond_traj"],            # base+traj_cat
          groups=["base", "traj"]),

    # ── Levels 3–5 — best L1 cond group (opp) × mechanism ───────────────────
    _spec(arch="film",                                         # base+opp_film
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"]),
    _spec(arch="film",                                         # base+opp_film_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          cond_dropout_p=0.1),
    _spec(arch="film",                                         # base+opp_film_four
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True),
    _spec(arch="film",                                         # base+opp_film_four_guid
          consumed_keys=["cond_base", "cond_opp"],
          groups=["base", "opp"],
          fourier=True,
          cond_dropout_p=0.1),

    # ── Trajectory conditioning, and the opp × traj pair ────────────────────
    _spec(arch="film",                                         # base+traj_film_four
          consumed_keys=["cond_base", "cond_traj"],
          groups=["base", "traj"],
          fourier=True),
    _spec(arch="film",                                         # base+opp+traj_film_four
          consumed_keys=["cond_base", "cond_opp", "cond_traj"],
          groups=["base", "opp", "traj"],
          fourier=True),
    # Same as base+traj_film_four plus the window-validity mask — the "v".
    _spec(arch="film",                                         # base+trajv_film_four
          consumed_keys=["cond_base", "cond_traj", "cond_traj_valid"],
          groups=["base", "traj"],
          fourier=True),
]

# Keys the registry by canonical name; raises if two specs collide, which
# catches both duplicates and knobs the naming scheme doesn't yet encode.
EXPERIMENTS_EMP = build_experiment_registry(_SPECS_EMP, _BASE_TEMPLATE)

for name, cfg in EXPERIMENTS_EMP.items():
    print(f"{name:26s}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")


## wandb setup

Same per-student project convention as `11_train_and_evaluate.ipynb`. The
empirical runs go to a sibling project (suffix `-emp`) so the residual and
empirical dashboards stay separable; switch to the same project as the
residual notebook if you want them side-by-side.

If wandb is unavailable, `train_empirical_experiment` falls back to a local
`CSVLogger` under `CKPT_DIR/csv_logs/`.

In [ ]:
WANDB_PROJECT = "butterflai-w10ext-amunoz-emp"
WANDB_ENTITY  = None

## Training

`train_empirical_experiment` is idempotent: if `ckpt_emp_<name>.ckpt` already
exists in `CKPT_DIR`, it skips training. Delete the file (or change the
experiment name) to force a retrain.

Training uses the same DDIM cosine schedule and AdamW optimizer as the
residual notebook; the only loss-side difference is the additive
`KL(p_model || p_classical)` term computed by recovering the predicted clean
logits, softmaxing, and comparing against the per-window classical mass
carried in the batch (`m_classical`).

In [ ]:
# EDIT THIS LIST. train_empirical_experiment is idempotent: it skips any
# experiment whose ckpt_emp_<name>.ckpt already exists.
#
# HEADS-UP after the rename: all 18 experiments were trained under the OLD
# E<n>metrics stems, and those files were deliberately NOT renamed. The skip
# keys off ckpt_emp_<canonical-name>.ckpt, so anything enabled here RETRAINS
# FROM SCRATCH even though a legacy checkpoint exists. 11d and 11e resolve the
# legacy files through resolve_experiment_stems(), so there is nothing to
# re-run just to get a scoreboard. Enable a name here only when you want fresh
# weights (or `git mv` its ckpt_emp_E<n>metrics.ckpt to the canonical stem).
#
# Base conditioning:  base_cat  base_film  base_cat_four  base_film_four
#                     base_cat_guid  base_film_guid  base_cat_four_guid
#                     base_film_four_guid
# Added conditioning: base+hemi_cat  base+opp_cat  base+traj_cat
#                     base+opp_film  base+opp_film_guid  base+opp_film_four
#                     base+opp_film_four_guid  base+traj_film_four
#                     base+opp+traj_film_four  base+trajv_film_four

ENABLED_EXPERIMENTS_EMP = []

for _name in ENABLED_EXPERIMENTS_EMP:
    if _name not in EXPERIMENTS_EMP:
        raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS_EMP)}")
    print(f"\n=== training {_name} ===")
    train_empirical_experiment(
        name=_name, cfg=EXPERIMENTS_EMP[_name], windows_aug=windows_aug,
        classical=classical, bin_centers=BIN_CENTERS,
        ckpt_dir=CKPT_DIR, alpha_np=alpha_np, sigma_np=sigma_np, T=T,
        bin_width=BIN_WIDTH,
        wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
    )

# Every checkpoint on disk that maps to a spec, canonical or pre-rename.
_ckpts_emp = discover_emp_experiment_checkpoints(CKPT_DIR)
TRAINED, _unknown_emp = resolve_experiment_stems(_ckpts_emp, EXPERIMENTS_EMP)
if _unknown_emp:
    print(f"\ncheckpoints with no spec (ignored): {_unknown_emp}")
print(f"\ntrained empirical checkpoints: {[c for c, _ in TRAINED]}")


## Quick-look diagnostic

For each trained variant, sample K=20 densities on one val window and plot
their mean against the classical density at that window. Three failure
modes to look for:

- Sample mean **pinned to classical** → `lambda_kl` is too strong; lower it.
- Sample mean **near uniform** → KL too strong or training under-converged.
- Sample mean **wildly different from classical with high variance across K** →
  KL too weak / training noise dominates. Raise `lambda_kl` or `max_epochs`.

The simplex check at the top of the cell is the architectural guarantee:
every sample is non-negative and integrates to 1 to float32 precision.

In [ ]:
import scipy.stats as _sst

if not TRAINED:
    print("no trained checkpoints found — run the training cell above first")
else:
    n_show = len(TRAINED)
    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4), squeeze=False)
    axes = axes[0]

    # TRAINED is [(canonical name, on-disk stem)] — load by stem, label by name.
    for ax, (name, stem) in zip(axes, TRAINED):
        cfg = EXPERIMENTS_EMP[name]
        lit, train_ds, val_ds, total_dim = load_trained_empirical_experiment(
            name=stem, cfg=cfg, windows_aug=windows_aug,
            classical=classical, bin_centers=BIN_CENTERS,
            ckpt_dir=CKPT_DIR, alpha_np=alpha_np, sigma_np=sigma_np,
            bin_width=BIN_WIDTH,
        )

        idx = 0
        cond_one = torch.cat(
            [val_ds[idx][k] for k in cfg["consumed_keys"]], dim=-1
        ).unsqueeze(0).repeat(20, 1)
        torch.manual_seed(0)
        p_samples = sample_empirical_extended(
            lit, cond_one, guidance_w=0.0, bin_width=BIN_WIDTH, device=device,
        ).cpu().numpy()

        nonneg = (p_samples >= 0).all()
        mass   = (p_samples * BIN_WIDTH).sum(axis=-1)
        print(f"{name}: non-negative={nonneg}  |Σp·Δℓ - 1|_max={np.abs(mass - 1).max():.2e}")

        m_cl = val_ds._m_classical[idx].numpy() / BIN_WIDTH  # mass -> density

        ax.plot(BIN_CENTERS, p_samples.mean(axis=0), "C1-", lw=2, label=f"{name} mean")
        ax.fill_between(
            BIN_CENTERS,
            p_samples.mean(axis=0) - p_samples.std(axis=0),
            p_samples.mean(axis=0) + p_samples.std(axis=0),
            color="C1", alpha=0.2, label="±1σ across K",
        )
        ax.plot(BIN_CENTERS, m_cl, "C0--", lw=1.5, label="classical")
        ax.set_xlabel("|latitude| (°)")
        ax.set_ylabel("density")
        ax.set_title(f"{name}: val[{idx}]")
        ax.legend(fontsize=8)

    fig.suptitle("Empirical-diffusion diagnostic: decoded densities vs classical")
    fig.tight_layout()
    plt.show()